In [1]:
!pip install -U langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13


In [2]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.2 MB/s eta 0:00:00


In [3]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")

model = init_chat_model(
    model="groq:llama-3.3-70b-versatile",
    api_key=api_key
)

In [4]:
!pip install -U langchain-tavily

In [5]:
tavily_api_key = userdata.get("TAVILY_API")

In [6]:
from langchain_tavily import TavilySearch
from pprint import pprint

search_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    topic = "general",
    tavily_api_key=tavily_api_key

)


In [8]:
response = search_tool.invoke({'query':"Who is Modi"})
pprint(response)

{'answer': None,
 'follow_up_questions': None,
 'images': [],
 'query': 'Who is Modi',
 'request_id': '42107a04-c6a9-450d-9977-deaaf5aa7c72',
 'response_time': 4.47,
 'results': [{'content': 'Narendra Modi (born September 17, 1950, Vadnagar, '
                         'India) is an Indian politician and government '
                         'official who rose to become a senior leader of the '
                         'Bharatiya Janata Party (BJP). In 2014 he led his '
                         'party to victory in elections to the Lok Sabha '
                         '(lower chamber of the Indian parliament), after '
                         'which he was sworn in as prime minister of India. '
                         'Prior to that he had served (2001–14) as chief '
                         'minister (head of government) of Gujarat state in '
                         'western India. The Modi-led BJP (and the BJP-led NDA '
                         'alliance) won the majority again in t

In [22]:
import requests
from langchain.tools import tool

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    rapidapi_key = userdata.get('RAPID_API')

    url = "https://jsearch.p.rapidapi.com/search-v2"
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
    }
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    print(response.status_code)
    print(data)


    jobs = data.get("data", {}).get("jobs", [])[:3]  # Only return top 3
    print(f"Found {len(jobs)} jobs\n")

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city")
        })
    return result

In [10]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [24]:
from langchain.agents import create_agent
agent = create_agent(
    model=model,
    tools=[search_jobs,search_tool],
    system_prompt=system_prompt
)

In [25]:
user_query = "What's the demand for Software Engineer role in the industry and show me related job openings in India"

response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})




Calling search_jobs tool
Searching jobs for: Software Engineer in India
200
{'status': 'OK', 'request_id': 'c692e65b-2800-444f-9990-78a3d5d8d763', 'parameters': {'query': 'Software Engineer in India', 'num_pages': 1, 'country': 'in', 'language': 'en'}, 'data': {'jobs': [{'job_id': 'z5jrAfeE_VsoMLCoAAAAAA==', 'job_title': 'Software Engineer (India)', 'employer_name': 'Ametek, Inc.', 'employer_logo': 'https://t0.gstatic.com/faviconV2?client=SOCIAL&type=FAVICON&fallback_opts=TYPE,SIZE,URL&url=https://ametek.com&size=128', 'employer_website': 'https://www.ametek.com', 'job_publisher': 'Ametek Careers', 'job_employment_type': 'Full–time', 'job_employment_types': ['FULLTIME'], 'job_apply_link': 'https://jobs.ametek.com/job/Bangalore-Software-Engineer-%28India%29-KA-560048/1410431200/', 'job_apply_is_direct': False, 'apply_options': [{'apply_link': 'https://jobs.ametek.com/job/Bangalore-Software-Engineer-%28India%29-KA-560048/1410431200/', 'is_direct': False, 'publisher': 'Ametek Careers'}],

In [26]:
pprint(response["messages"][-1].content)

('The demand for software engineers is increasing due to the widespread '
 'adoption of mobile devices and apps, growing need for medical software, '
 'expanding AI and automation applications, enhanced focus on cybersecurity, '
 'and continuous technological innovations. The job outlook for software '
 'engineering roles over the next decade is projected to grow 25% from '
 '2022-2032, which is much faster than the average across all occupations. '
 'This translates to approximately 153,900 new job openings for software '
 'engineers and related roles each year over the next decade.\n'
 '\n'
 'Some of the key skills in demand for software engineers include programming '
 'languages such as Python, JavaScript, Java, Go, and Rust, as well as cloud '
 'computing, AI, and machine learning fundamentals. DevOps and CI/CD are also '
 'critical knowledge areas for software engineers.\n'
 '\n'
 'The impact of AI, cloud, and automation on the software engineering job '
 'market is positive, wit